# Week 1: Illustrative food-delivery batch assignment

## Objective
Show how a small combinatorial optimization can balance delivery workload across hypothetical batches. This is an educational example only: it is **not** a production vehicle-routing model and it does not produce a real-world route recommendation.

## Data and selected deliveries
The script reads `data/cleaned_logistics_data.csv`, accepts only records with validated restaurant and delivery coordinates, and selects 12 real records from the most populated rounded 0.1° destination tile. The tile is used only to create a geographically compact teaching example; it is not an official service area. The input duration is the historical `time_taken_min` value, used as a **hypothetical workload estimate**. It is a realized outcome with hindsight, not a prospective dispatch prediction. Approximate distance and multiple-delivery status are shown as context, but no route-distance/travel-time matrix is available to model route interactions.

## Optimization formulation
Let `x[i,j]` indicate whether delivery `i` is assigned to batch/vehicle `j`. Every delivery is assigned exactly once, each of three batches can hold at most four deliveries, and the objective minimizes the maximum sum of estimated duration in any batch (the makespan). A baseline assigns sampled orders in sequence in groups of four. The optimized solution is found by exact enumeration of feasible assignments; 12 orders are small enough for a transparent brute-force search.

Why makespan? With fixed independent delivery times, the sum across all individual deliveries is constant regardless of assignment. For three parallel batches, balancing batch workloads can lower the estimated time until all batches finish. Thus the reported improvement is in this completion-time proxy, not in total individual service duration.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src/optimization_demo.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from optimization_demo import run_demo
result=run_demo(ROOT/'data/cleaned_logistics_data.csv',make_figures=True)
metrics=pd.Series({'Deliveries':result['number_of_deliveries'],'Batches/vehicles':result['number_of_batches'],'Capacity per batch':result['capacity_per_batch'],'Baseline makespan proxy (min)':result['baseline_makespan_min'],'Optimized makespan proxy (min)':result['optimized_makespan_min'],'Improvement (min)':result['improvement_min'],'Improvement (%)':result['improvement_percent'],'Total individual estimated work (min; unchanged)':result['total_estimated_work_min']})
display(metrics.to_frame('value').round(2))
print('Baseline batch workloads (minutes):',result['baseline_batch_loads'])
print('Optimized batch workloads (minutes):',result['optimized_batch_loads'])
orders=result['orders'].copy(); orders.insert(0,'demo_order',range(1,len(orders)+1))
display(orders[['demo_order','estimated_duration_min','approx_delivery_distance_km','multiple_deliveries','delivery_location_latitude','delivery_location_longitude','baseline_batch','optimized_batch']].round(2))
print('Figures saved in visualizations/')

                                                   value
Deliveries                                         12.00
Batches/vehicles                                    3.00
Capacity per batch                                  4.00
Baseline makespan proxy (min)                     116.00
Optimized makespan proxy (min)                    104.00
Improvement (min)                                  12.00
Improvement (%)                                    10.34
Total individual estimated work (min; unchanged)  310.00
Baseline batch workloads (minutes): [97.0, 116.0, 97.0]
Optimized batch workloads (minutes): [103.0, 104.0, 103.0]
    demo_order  estimated_duration_min  approx_delivery_distance_km  multiple_deliveries  delivery_location_latitude  delivery_location_longitude  baseline_batch  optimized_batch
0            1                    33.0                        13.99                  2.0                       12.42                        76.71               1                1
1            2

## Assumptions and interpretation
- Three identical vehicles/batches are available simultaneously; each can take at most four deliveries.
- Each historical duration is treated as an independent sequential workload unit; it does not account for drive-time overlap, route sequence, pickup waiting, handoff, or vehicle differences.
- All 12 selected orders must be assigned exactly once. There are no delivery time windows, driver shifts, or additional feasibility constraints.
- The baseline groups deliveries in sampled row order. The optimization is exact for this small objective and these assumptions, not for operational dispatch.
- Distance is approximate straight-line distance between restaurant and destination, not road distance. It is reported for context and is not used as a route cost.

The comparison demonstrates a useful logistics principle: assigning work to reduce the most-loaded vehicle can shorten a parallel operation's completion time even when the total underlying work is unchanged. The result does not prove that a real dispatch or routing policy would achieve the same savings.

## Visualization
The first chart compares baseline and optimized makespan. The second shows which hypothetical batch receives each delivery. Neither figure describes actual roads or routes.

![Baseline versus optimized completion time](../visualizations/optimization_makespan_comparison.png)

![Illustrative delivery batch assignments](../visualizations/optimization_batch_assignments.png)

## Limitations and how a real optimizer could grow
This dataset does not contain complete road-network routes, route distances, actual travel-time matrices, dispatch timing, route sequences, vehicle capacity, fleet availability, order release times, delivery windows, fleet size, driver shifts, or operating costs. Consequently, this example is not a real vehicle-routing problem and is not a dispatch recommendation.

A future optimization formulation could extend to:
- **Vehicle routing:** minimize network travel time with road-based travel matrices and stop sequences.
- **Driver assignment:** match orders or zones to available drivers and shifts.
- **Batching:** group compatible orders while modeling pickup readiness, route order, and added delay.
- **Delivery windows:** penalize or constrain late arrival against promised windows.
- **Fleet capacity:** enforce vehicle load, volume, and fleet-size constraints.
- **Route cost:** minimize fuel, distance, labor, or total operating cost.

Required data includes road-network travel times and distances, driver availability, vehicle capacity and fleet size, order release/ready times, promised delivery windows, driver shift constraints, route/stop sequences, and operating costs. Real service-area demand by time and operational rules are also needed.